QPE without QECC, purely on the physical level

In [1]:
from qiskit import __version__
print(__version__)

2.1.1


In [30]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, state_fidelity, Pauli, DensityMatrix, partial_trace
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_aer.library import SaveDensityMatrix
from qiskit import transpile 
import numpy as np
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.circuit.library import HGate, UnitaryGate, IGate
import matplotlib.pyplot as plt
import random
import math
import time
from typing import List

In [3]:
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))
amp_0 = np.cos(theta/2)
amp_1 = np.sin(theta/2)

# Infidelity to Error Rates

In [4]:
def inf_to_error(infidelity_list, num_qubits):
    error_rates_list = []
    dimension = 2**num_qubits
    for infidelity in infidelity_list:
        error_rates_list.append( infidelity * (dimension / (dimension - 1)) )
    
    return error_rates_list

In [5]:
one_qubit_gate_infidelity = [2.8e-5 * i for i in range(1,6)]
two_qubit_gate_infidelity = [8.3e-4 * i for i in range(1,6)]
idle_gate_infidelity = [1.2e-4 * i for i in range(1,6)]
spam0 = [6.7e-4 * i for i in range(1,6)] # P(1|0), measured 1 given that 0 was prepared
spam1 = [1.2e-3 * i for i in range(1,6)] # P(0|1)

In [6]:
one_qubit_gate_error_prob = inf_to_error(one_qubit_gate_infidelity, num_qubits=1)
two_qubit_gate_error_prob = inf_to_error(two_qubit_gate_infidelity, num_qubits=2)
idle_gate_error_prob = inf_to_error(idle_gate_infidelity, num_qubits=1)

# Functions

In [7]:
def state_prep(qc: QuantumCircuit, alpha0: float, alpha1: float):
    qc.h(0)
    qc.x(1)
    qc.ry(alpha0, 1)
    qc.rz(alpha1, 1)

In [8]:
def control_u(qc: QuantumCircuit, h1: float, h2: float, t: float):
    qc.rz(h1*t, 1)
    qc.cx(0,1)
    qc.rz(-h1*t, 1)
    qc.cx(0,1)
    qc.h(1)
    qc.rz(h2*t, 1)
    qc.cx(0,1)
    qc.rz(-h2*t, 1)
    qc.cx(0,1)
    qc.h(1)

In [35]:
def qpe(qc: QuantumCircuit, alpha0: float, alpha1: float, h1: float, h2: float, t: float, beta: float, meas_bit: ClassicalRegister, k: int):
    state_prep(qc, alpha0, alpha1)
    for _ in range(k):
        control_u(qc, h1, h2, t)
    
    qc.rz(beta, 0)
    qc.h(0)
    qc.measure(0, meas_bit)

In [18]:
seed = time.time_ns()
rng = random.Random(seed)

h1, h2, h3 = (0.79605, -0.18092, -0.32096)
alpha0, alpha1 = (-0.274220, -0.785398)
t = np.pi / (8*h1)
qc = QuantumCircuit(2)

In [75]:
kmax = 50
num_repeat_circuit = 500
meas_list = []
beta_list = []
k_list = []
for _ in range(num_repeat_circuit):
    noise_model = NoiseModel()

    meas_circuit = qc.copy()
    
    beta = rng.uniform(0, math.pi / 2)
    k = rng.randint(1, kmax)
    beta_list.append(beta)
    k_list.append(k)
    
    meas_bit = ClassicalRegister(1)
    meas_circuit.add_register(meas_bit)
    qpe(meas_circuit, alpha0, alpha1, h1, h2, t, beta, meas_bit, k)

    backend = AerSimulator(noise_model=noise_model, method='statevector')
    job = backend.run(meas_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    meas_list.append(int(list(counts.keys())[0]))

In [76]:
def estimate_phase(meas_list: List[int], beta_list: List[float], k_list: List[int], phi_list: np.ndarray):
    def prob_of_meas_outcome(meas, beta, k, phi):
        return (1 + np.cos(k*phi + beta - meas*np.pi)) / 2

    
    Q_list = []
    for phi in phi_list:
        temp = 1
        for i in range(num_repeat_circuit):
            temp *= prob_of_meas_outcome(meas_list[i], beta_list[i], k_list[i], phi)
        Q_list.append(temp)
        
    max_val = max(Q_list)
    return max_val, Q_list.index(max_val)

In [77]:
phi_list = np.linspace(0, 2*np.pi, 50)
max_val, idx = estimate_phase(meas_list, beta_list, k_list, phi_list)
ground_state_energy = -max_val/t

In [78]:
print(max_val)
print(idx)
print(ground_state_energy)

1.5449131204307503e-118
3
-3.1317315136031153e-118


In [82]:
H = h1*np.array([[1,0], [0,-1]]) + h2*np.array([[0,1], [1,0]]) + h3*np.eye(2, dtype='complex')

eigenvalues, eigenvectors = np.linalg.eigh(H)
actual_ground_state_energy = eigenvalues[0]
print(actual_ground_state_energy)

-1.137310199914228
